In [1]:
import hashlib


def hash_leaf(leaf: str) -> str:
    # return f'h({leaf})'
    return hashlib.sha256(leaf.encode()).hexdigest()


def hash_pair(left: str, right: str) -> str:
    # return f'h({left}, {right})'
    return hashlib.sha256((left + right).encode()).hexdigest()


def cut(s: str) -> str:
    return s[:5] + "..."


def calc_root(hashes: list[str]) -> str:
    n = len(hashes)
    assert n > 0

    hs = hashes[:]

    while n > 1:
        for i in range(0, n, 2):
            left = hs[i]
            right = hs[min(i + 1, n - 1)]
            hs[i // 2] = hash_pair(left, right)
        n = (n + 1) // 2

    return hs[0]


def get_proof(hashes: list[str], idx: int) -> list[str]:
    proof = []
    n = len(hashes)
    assert n > 0

    hs = hashes[:]

    while n > 1:
        if idx & 1:
            j = idx - 1
        else:
            j = min(idx + 1, n - 1)

        proof.append(hs[j])
        idx //= 2

        for i in range(0, n, 2):
            left = hs[i]
            right = hs[min(i + 1, n - 1)]
            hs[i // 2] = hash_pair(left, right)

        n = (n + 1) // 2

    return proof


def verify(root: str, proof: list[str], hashes: list[str], idx: int) -> bool:
    h = hashes[idx]

    for p in proof:
        if idx & 1 == 0:
            left, right = h, p
        else:
            left, right = p, h

        h = hash_pair(left, right)
        idx //= 2

    return h == root


vals = ["A", "B", "C", "D", "E", "F", "G"]
hashes = [hash_leaf(v) for v in vals]
root = calc_root(hashes)

for i in range(len(vals)):
    proof = get_proof(hashes, i)
    assert verify(root, proof, hashes, i)